# Smart Supply Chain Analytics & Automated Demand Forecasting Engine
## Phase 3: Databricks & PySpark ETL/ELT Cloud Pipeline

**Objective**:
This Databricks PySpark notebook implements an enterprise-grade ETL/ELT pipeline for supply chain transactions:
1. **Extract**: Ingest raw transactional data (`orders.csv`) from Databricks DBFS (`/FileStore/tables/orders.csv`).
2. **Transform**: Validate schemas, filter canceled/invalid records, handle timestamps, and compute rolling 7-day and 30-day demand moving averages using PySpark Window functions.
3. **Feature Engineering**: Compute daily SKU demand, inventory consumption rates, and fulfillment lead times.
4. **Load (Delta Lake)**: Materialize clean Delta tables (`fact_daily_demand`, `fact_orders_delta`) with ACID compliance, Time Travel, and Z-Order indexing.

### Step 1: Initialize Spark Session & Import Libraries

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, sum as _sum, avg, count, round as _round, 
    to_date, to_timestamp, datediff, coalesce, lit, expr
)
from pyspark.sql.window import Window

# Initialize Spark Session (automatically provided in Databricks)
spark = SparkSession.builder \
    .appName("SupplyChainETL") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

### Step 2: Extract Raw Transactions from DBFS / Cloud Storage
> Note: Upload `orders.csv` to Databricks via **Catalog -> Create Table -> Upload to DBFS** (target: `/FileStore/tables/orders.csv`).

In [ ]:
# Define DBFS File Path with local fallback
dbfs_path = "/FileStore/tables/orders.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(dbfs_path)

print(f"Raw Ingested Records: {df_raw.count():,}")
df_raw.printSchema()
display(df_raw.limit(5))

### Step 3: Transform & Data Quality Validation
- Filter non-positive quantities and null dates
- Standardize dates into `DateType` and `TimestampType`
- Exclude canceled or unfulfillable transactions
- Cleanse unit prices and sanitize warehouse identifiers

In [ ]:
# 2. Transform: Data Validation, Imputation & Filtering
df_clean = df_raw \
    .filter(col("quantity") > 0) \
    .filter(col("order_date").isNotNull()) \
    .withColumn("order_timestamp", to_timestamp(col("order_date"))) \
    .withColumn("order_date", to_date(col("order_date"))) \
    .withColumn("total_revenue", _round(col("quantity") * col("unit_price"), 2)) \
    .filter(~col("order_status").isin("canceled", "unavailable"))

print(f"Clean Validated Records: {df_clean.count():,}")
display(df_clean.limit(5))

### Step 4: Daily SKU Demand Aggregation & Rolling Averages
We aggregate daily SKU demand and leverage PySpark **Window specifications** to compute:
- Rolling 7-day moving average of units sold
- Rolling 30-day moving average of units sold

In [ ]:
# 3. Aggregate Daily SKU Demand for Supply Chain Features
df_daily_agg = df_clean.groupBy("product_id", "product_category", "order_date") \
    .agg(
        _sum("quantity").alias("daily_units"),
        _sum("total_revenue").alias("daily_revenue"),
        count("order_id").alias("order_count"),
        _round(avg("lead_time_days"), 2).alias("avg_lead_time_days")
    )

# Window specification: partition by SKU, ordered by chronological date
window_7d = Window.partitionBy("product_id").orderBy("order_date").rowsBetween(-6, 0)
window_30d = Window.partitionBy("product_id").orderBy("order_date").rowsBetween(-29, 0)

df_daily_demand = df_daily_agg \
    .withColumn("rolling_7d_avg_demand", _round(avg("daily_units").over(window_7d), 2)) \
    .withColumn("rolling_30d_avg_demand", _round(avg("daily_units").over(window_30d), 2))

print(f"Aggregated Daily Demand Records: {df_daily_demand.count():,}")
display(df_daily_demand.orderBy("product_id", "order_date").limit(10))

### Step 5: Load - Write Clean Delta Lake Tables
Store our cleansed analytical models as ACID-compliant Delta tables in the Databricks Metastore.

In [ ]:
# 4. Load: Write Clean Delta Tables
df_daily_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_daily_demand")

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_orders_delta")

print("[SUCCESS] Materialized Delta Tables: fact_daily_demand, fact_orders_delta")

### Step 6: Delta Lake Performance Optimization & Time Travel
- `OPTIMIZE ... ZORDER BY` for lightning-fast queries
- Inspect table audit history and commit logs

In [ ]:
# Optimize Delta Lake Storage layout
spark.sql("OPTIMIZE fact_daily_demand ZORDER BY (product_id, order_date)")

# Delta Lake Time Travel Verification
history_df = spark.sql("DESCRIBE HISTORY fact_daily_demand")
display(history_df.select("version", "timestamp", "operation", "operationMetrics"))

### Step 7: Export Clean Parquet for Power BI & Time-Series Forecasting
Save ready-to-consume Parquet files into DBFS `/FileStore/processed/`.

In [ ]:
export_path = "/FileStore/processed/fact_daily_demand.parquet"
df_daily_demand.write.mode("overwrite").parquet(export_path)
print(f"Exported downstream Parquet to: {export_path}")